# 🟣 Getting Started with ChromaDB

**Day 6 — Notebook 2 of 4 | Estimated Time: 30 minutes**

---

## What You'll Learn
- Create and manage ChromaDB collections
- Add, update, delete, and retrieve documents (full CRUD)
- Query by semantic similarity using real Gemini embeddings
- Filter results by metadata
- Persist a collection to disk

---

## Setup

In [ ]:
%pip install -U -q "google-genai>=1.0.0" chromadb

In [ ]:
import sys, os
import numpy as np
import chromadb
from chromadb.config import Settings

repo_root = os.path.abspath(os.path.join(os.getcwd(), "../../.."))
if repo_root not in sys.path:
    sys.path.append(repo_root)

from helpers.auth import get_api_key
from google import genai
from google.genai import types

client = genai.Client(api_key=get_api_key())
EMBEDDING_MODEL = "text-embedding-004"
print("✅ Ready!")

---

## 1. ChromaDB Client Modes

ChromaDB supports two modes:

| Mode | Use When |
|------|----------|
| `chromadb.Client()` | In-memory — data lost when Python exits. Good for testing. |
| `chromadb.PersistentClient(path)` | Saves to disk — data survives restarts. Good for development. |

In [ ]:
# In-memory client (for this notebook)
chroma = chromadb.Client()
print("✅ In-memory ChromaDB client created.")

# Persistent client (uncomment to use disk storage)
# chroma = chromadb.PersistentClient(path="/tmp/my_chroma_db")
# print("✅ Persistent ChromaDB client created at /tmp/my_chroma_db")

---

## 2. Creating Collections

A **collection** is ChromaDB's equivalent of a table. You store related documents in a collection.

In [ ]:
# Create a collection
collection = chroma.create_collection(
    name="company_knowledge_base",
    metadata={
        "hnsw:space": "cosine",   # Distance metric (cosine, l2, ip)
        "description": "Company internal FAQs and policies",
    },
)
print(f"Created collection: {collection.name}")
print(f"Document count: {collection.count()}")

In [ ]:
# get_or_create_collection — safe for repeated runs
# If collection exists, returns it; if not, creates it
col2 = chroma.get_or_create_collection(
    name="company_knowledge_base",
    metadata={"hnsw:space": "cosine"},
)
print(f"Got collection: {col2.name} (same as before: {col2 is not None})")

# List all collections
print(f"\nAll collections: {[c.name for c in chroma.list_collections()]}")

---

## 3. A Helper to Embed with Gemini

ChromaDB doesn't call Gemini automatically — we need to embed text ourselves and pass the vectors.

In [ ]:
def embed_texts(texts: list[str], task_type: str = "RETRIEVAL_DOCUMENT") -> list[list[float]]:
    """Embed a list of texts using Gemini and return as list of lists."""
    result = client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=texts,
        config=types.EmbedContentConfig(task_type=task_type),
    )
    return [e.values for e in result.embeddings]


# Quick test
test_vec = embed_texts(["hello world"])
print(f"Embedding dimensions: {len(test_vec[0])}")

---

## 4. Adding Documents (Create)

In [ ]:
# Sample documents with rich metadata
documents = [
    {"id": "hr-001", "text": "Full-time employees receive 25 days annual leave per year. Leave must be approved by your manager at least 2 weeks in advance.", "category": "HR", "topic": "leave"},
    {"id": "hr-002", "text": "Employees may work remotely up to 3 days per week. At least 2 days in the office are required each week.", "category": "HR", "topic": "remote-work"},
    {"id": "hr-003", "text": "Primary caregivers receive 26 weeks of fully paid parental leave. Secondary caregivers receive 4 weeks.", "category": "HR", "topic": "leave"},
    {"id": "it-001", "text": "Install the GlobalProtect VPN client to access company systems remotely. Use your company email and password.", "category": "IT", "topic": "access"},
    {"id": "it-002", "text": "Passwords must be at least 12 characters with uppercase, lowercase, numbers, and symbols. Passwords expire every 90 days.", "category": "IT", "topic": "security"},
    {"id": "it-003", "text": "To request new software, submit a ticket at it.company.com. Requests under £100/year are auto-approved. Processing takes 3-5 business days.", "category": "IT", "topic": "software"},
    {"id": "fin-001", "text": "Submit expense claims within 30 days of the expense via the Expenses app. All claims require receipts. Meals are reimbursed up to £40/day when travelling.", "category": "Finance", "topic": "expenses"},
    {"id": "eng-001", "text": "All code changes require at least 2 approvals before merging. One approval must be from a senior engineer or tech lead.", "category": "Engineering", "topic": "process"},
]

texts = [d["text"] for d in documents]
ids = [d["id"] for d in documents]
metadatas = [{"category": d["category"], "topic": d["topic"]} for d in documents]

# Generate embeddings
print("Embedding documents...")
embeddings = embed_texts(texts)

# Add to ChromaDB
collection.add(
    ids=ids,
    documents=texts,
    embeddings=embeddings,
    metadatas=metadatas,
)

print(f"✅ Added {len(documents)} documents. Collection size: {collection.count()}")

---

## 5. Querying (Semantic Search)

In [ ]:
def search(query: str, n_results: int = 3, where: dict = None) -> None:
    """Query the collection and print formatted results."""
    query_embedding = embed_texts([query], task_type="RETRIEVAL_QUERY")[0]
    
    kwargs = {
        "query_embeddings": [query_embedding],
        "n_results": n_results,
        "include": ["documents", "metadatas", "distances"],
    }
    if where:
        kwargs["where"] = where
    
    results = collection.query(**kwargs)
    
    print(f"🔍 Query: '{query}'")
    if where:
        print(f"   Filter: {where}")
    
    for i in range(len(results["ids"][0])):
        doc_id = results["ids"][0][i]
        doc_text = results["documents"][0][i]
        meta = results["metadatas"][0][i]
        dist = results["distances"][0][i]
        similarity = 1 - dist  # cosine distance → cosine similarity
        print(f"  {i+1}. [{similarity:.3f}] ({meta['category']}) {doc_text[:80]}...")
    print()


# Run some searches
search("How many days of holiday do I get?")
search("I can't connect to the company network from home.")
search("How do I get a new tool for my team?")

---

## 6. Metadata Filtering

Use `where` to filter results by metadata **before** running ANN search. This is much faster than filtering after retrieval.

In [ ]:
# Filter by category
search("What are the rules?", n_results=3, where={"category": "IT"})

# Filter by topic
search("Time off work", n_results=3, where={"topic": "leave"})

# Combine filters with $and
search("What do I need to know?", n_results=2, where={
    "$and": [
        {"category": {"$eq": "IT"}},
        {"topic": {"$eq": "security"}},
    ]
})

In [ ]:
# ChromaDB filter operators reference
filter_examples = {
    "exact match": {"category": "IT"},
    "not equal": {"category": {"$ne": "HR"}},
    "in list": {"category": {"$in": ["HR", "Finance"]}},
    "not in list": {"category": {"$nin": ["Engineering"]}},
    "AND": {"$and": [{"category": "IT"}, {"topic": "security"}]},
    "OR": {"$or": [{"category": "HR"}, {"category": "Finance"}]},
}

print("ChromaDB Metadata Filter Operators:\n")
for name, filt in filter_examples.items():
    print(f"  {name}: {filt}")

---

## 7. Get, Update, Delete (Full CRUD)

In [ ]:
# READ — get a specific document by ID
result = collection.get(
    ids=["hr-001"],
    include=["documents", "metadatas"],
)
print("GET hr-001:")
print(f"  Text: {result['documents'][0][:80]}...")
print(f"  Metadata: {result['metadatas'][0]}")
print()

In [ ]:
# UPDATE — update an existing document
updated_text = "Full-time employees receive 28 days annual leave per year (updated policy 2025). Leave must be approved by your manager at least 2 weeks in advance."
updated_embedding = embed_texts([updated_text])[0]

collection.update(
    ids=["hr-001"],
    documents=[updated_text],
    embeddings=[updated_embedding],
    metadatas=[{"category": "HR", "topic": "leave", "version": "2025"}],
)

# Verify the update
result = collection.get(ids=["hr-001"], include=["documents", "metadatas"])
print("UPDATED hr-001:")
print(f"  Text: {result['documents'][0][:90]}...")
print(f"  Metadata: {result['metadatas'][0]}")
print()

In [ ]:
# DELETE — remove a document
print(f"Count before delete: {collection.count()}")

collection.delete(ids=["eng-001"])

print(f"Count after delete:  {collection.count()}")

# Delete by metadata filter
# collection.delete(where={"category": "Finance"})  # Deletes all Finance docs

---

## 8. Persistent Storage

In [ ]:
import tempfile

# Save to disk
persist_path = "/tmp/day6_chroma_db"
os.makedirs(persist_path, exist_ok=True)

persist_client = chromadb.PersistentClient(path=persist_path)
persist_col = persist_client.get_or_create_collection(
    name="persistent_test",
    metadata={"hnsw:space": "cosine"},
)

# Add a document
persist_col.add(
    ids=["test-1"],
    documents=["This data will survive a restart."],
    embeddings=embed_texts(["This data will survive a restart."]),
)

print(f"Saved {persist_col.count()} document(s) to {persist_path}")

# Simulate restart: open a new client pointing to same path
del persist_client
reload_client = chromadb.PersistentClient(path=persist_path)
reloaded = reload_client.get_collection("persistent_test")
print(f"After 'restart': collection has {reloaded.count()} document(s) ✅")

---

## 🏋️ Exercise 1: Build and Query Your Own Collection

Create a collection on a topic of your choice and query it.

In [ ]:
# Exercise 1: Build your own collection
# TODO:
# 1. Choose a topic (recipes, book summaries, movie plots, programming concepts, etc.)
# 2. Create a new ChromaDB collection
# 3. Add at least 6 documents with meaningful metadata
# 4. Run at least 3 different search queries
# 5. Run at least 1 metadata-filtered search

my_collection = chroma.get_or_create_collection(
    name="my_topic",
    metadata={"hnsw:space": "cosine"},
)

# TODO: Add your documents
my_docs = [
    # {"id": "1", "text": "...", "category": "..."},
]

# TODO: Add to collection
# TODO: Search

---

## 🏋️ Exercise 2: upsert — Add or Update

ChromaDB has an `upsert` method that adds a document if it doesn't exist, or updates it if it does. Use it to build an idempotent document loader.

In [ ]:
# Exercise 2: Use upsert for idempotent loading
# TODO:
# 1. Use collection.upsert() instead of collection.add()
# 2. Call it twice with the same IDs but different text
# 3. Verify the second call updates rather than duplicates
# Hint: collection.upsert() has the same signature as collection.add()

upsert_col = chroma.get_or_create_collection("upsert_test", metadata={"hnsw:space": "cosine"})

doc_v1 = "The refund policy allows returns within 14 days."
doc_v2 = "The refund policy allows returns within 30 days (updated)."

# TODO: upsert v1, check count, upsert v2 with same ID, check count again
# Verify the document text changed but count stayed the same

---

## Key Takeaways

| Operation | ChromaDB Method |
|-----------|----------------|
| Create collection | `chroma.create_collection(name, metadata)` |
| Add documents | `collection.add(ids, documents, embeddings, metadatas)` |
| Search | `collection.query(query_embeddings, n_results, where)` |
| Get by ID | `collection.get(ids, include)` |
| Update | `collection.update(ids, documents, embeddings, metadatas)` |
| Upsert | `collection.upsert(...)` — add or update |
| Delete | `collection.delete(ids)` or `collection.delete(where=...)` |
| Persist | `chromadb.PersistentClient(path=...)` |

---

## 📚 Further Reading

| Resource | Type | Link |
|----------|------|------|
| ChromaDB Docs | Docs | [docs.trychroma.com](https://docs.trychroma.com/) |
| ChromaDB Filtering | Docs | [docs.trychroma.com/usage-guide#filtering-by-metadata](https://docs.trychroma.com/usage-guide#filtering-by-metadata) |

---

**Next up:** [03_Document_Chunking_Strategies.ipynb](./03_Document_Chunking_Strategies.ipynb)